In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
)

# Configuration
MODEL_PATH = "models/best_waste_cnn.keras"
TEST_DIR = "C:\Users\admin\Desktop\waste\final_merged_dataset"  
BATCH_SIZE = 32

RESCALE = 1.0 / 255.0

# model
model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
model.summary()

# 3. Image size taken from the model itself
img_height = model.input_shape[1]
img_width = model.input_shape[2]

# 4. Load test dataset (only ONCE)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(img_height, img_width),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,          # must stay False so file_paths match predictions
)

class_names = test_ds.class_names
file_paths = test_ds.file_paths   # FIX: save BEFORE .map(), because map() removes this attribute

print("\nClasses:", class_names)

# Preprocessing
test_ds = test_ds.map(
    lambda x, y: (x * RESCALE, y)
).prefetch(tf.data.AUTOTUNE)

# Evaluate model
results = model.evaluate(test_ds, verbose=1)

print("\nEvaluation Results:")
for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value:.4f}")

# Actual labels
y_true = np.concatenate([labels.numpy() for _, labels in test_ds])

# Predictions
predictions = model.predict(test_ds, verbose=1)

if predictions.ndim == 1 or predictions.shape[-1] == 1:
    probabilities = predictions.reshape(-1)
    y_pred = (probabilities >= 0.5).astype(int)
else:
    probabilities = predictions
    y_pred = np.argmax(probabilities, axis=1)

# Accuracy
accuracy = accuracy_score(y_true, y_pred)
print("\nTest Accuracy:", round(accuracy * 100, 2), "%")

#  Classification reporttttttttttt balle balle
print("\nClassification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0,
))

# 11. Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("Waste Classification - Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

# Incorrect predictions (uses the file_paths saved earlier)
errors = [
    {
        "Image": file_paths[i],
        "Actual": class_names[y_true[i]],
        "Predicted": class_names[y_pred[i]],
        "Confidence": float(np.max(probabilities[i])) if probabilities.ndim > 1 else float(probabilities[i]),
    }
    for i in range(len(y_true))
    if y_true[i] != y_pred[i]
]

error_df = pd.DataFrame(errors)
error_df.to_csv("incorrect_predictions.csv", index=False)

print("\nTotal test images:", len(y_true))
print("Incorrect predictions:", len(errors))
print("Correct predictions:", len(y_true) - len(errors))

print("\nSaved:")
print("- confusion_matrix.png")
print("- incorrect_predictions.csv")

Model loaded successfully!
Input shape: (None, 128, 128, 3)
Output shape: (None, 6)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 331,604 (1.26 MB)

 Trainable params: 110,534 (431.77 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 221,070 (863.56 KB)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset-resized'